# Standard Lyapunov MPC With First-Step Contraction

This notebook uses the maintained `Lyapunov/` implementation path. It computes the refined target selector each step and solves a standard tracking Lyapunov MPC with one additional hard contraction constraint on the first predicted step only.

In [ ]:
import os
import numpy as np

from Simulation.system_functions import PolymerCSTR
from Simulation.mpc import compute_observer_gain
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic
from utils.td3_helpers import load_and_prepare_system_data
from utils.scaling_helpers import apply_min_max
from Lyapunov.lyapunov_core import (
    design_standard_tracking_terminal_ingredients,
    FirstStepContractionTrackingLyapunovMpcSolver,
)
from Lyapunov.run_lyap_mpc import run_standard_tracking_lyapunov_mpc_first_step_contraction


In [ ]:
# Polymer CSTR setup
Ad = 2.142e17
Ed = 14897
Ap = 3.816e10
Ep = 3557
At = 4.50e12
Et = 843
fi = 0.6
m_delta_H_r = -6.99e4
hA = 1.05e6
rhocp = 1506
rhoccpc = 4043
Mm = 104.14
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

CIf = 0.5888
CMf = 8.6981
Qi = 108.0
Qs = 459.0
Tf = 330.0
Tcf = 295.0
V = 3000.0
Vc = 3312.4
system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

Qm_ss = 378.0
Qc_ss = 471.6
system_steady_state_inputs = np.array([Qc_ss, Qm_ss])
delta_t = 0.5

steady_states = {"ss_inputs": system_steady_state_inputs.copy()}
cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t, deviation_form=False)
steady_states["y_ss"] = cstr_ss.y_ss.copy()

u_min = np.array([71.6, 78.0])
u_max = np.array([870.0, 670.0])
setpoint_y_range_phys = np.array([[2.8, 320.0], [5.0, 326.0]])
setpoint_y_phys = np.array([[4.5, 324.0], [3.4, 321.0]])

system_data = load_and_prepare_system_data(
    steady_states=steady_states,
    setpoint_y=setpoint_y_range_phys,
    u_min=u_min,
    u_max=u_max,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]

inputs_number = int(B_aug.shape[1])
y_sp_scenario = apply_min_max(setpoint_y_phys, data_min[inputs_number:], data_max[inputs_number:]) - apply_min_max(
    steady_states["y_ss"],
    data_min[inputs_number:],
    data_max[inputs_number:],
)

n_tests = 200
set_points_len = 400
TEST_CYCLE = [False, False]

predict_h = 9
cont_h = 3

u_ss = apply_min_max(steady_states["ss_inputs"], data_min[:inputs_number], data_max[:inputs_number])
b_min = apply_min_max(u_min, data_min[:inputs_number], data_max[:inputs_number])
b_max = apply_min_max(u_max, data_min[:inputs_number], data_max[:inputs_number])
b1 = (b_min[0] - u_ss[0], b_max[0] - u_ss[0])
b2 = (b_min[1] - u_ss[1], b_max[1] - u_ss[1])
bnds = (b1, b2) * cont_h
IC_opt = np.zeros(inputs_number * cont_h)

Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0

Qy_diag_std = np.array([Q1_penalty, Q2_penalty], dtype=float)
Su_diag_std = np.array([0.0, 0.0], dtype=float)
Rdu_diag_std = np.array([R1_penalty, R2_penalty], dtype=float)

P_x, K_x, term_dbg = design_standard_tracking_terminal_ingredients(
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Qy_diag=Qy_diag_std,
    Su_diag=Su_diag_std,
    u_min=np.array([b1[0], b2[0]], dtype=float),
    u_max=np.array([b1[1], b2[1]], dtype=float),
    lambda_u=1.0,
    return_debug=True,
)

observer_poles = np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                           0.42900673, 0.4228262, 0.96916776, 0.91230187])
L = compute_observer_gain(A_aug, C_aug, observer_poles)

reward_config, reward_fn = make_reward_fn_mpc_quadratic(
    Q_diag=np.array([Q1_penalty, Q2_penalty]),
    R_diag=np.array([R1_penalty, R2_penalty]),
)

run_config = {
    "terminal_set_on": False,
    "terminal_alpha_scale": 0.0,
    "terminal_cost_scale": 0.005,
    "use_external_target_for_tracking": False,
    "use_target_on_solver_fail": False,
    "disturbance_after_step": True,
    "skip_terminal_if_alpha_small": True,
    "alpha_terminal_min": 1e-8,
    "rho_lyap": 0.99,
    "eps_lyap": 1e-9,
    "first_step_contraction_on": True,
    "Qs_tgt_diag": np.array([100.0, 100.0], dtype=float),
    "Ru_tgt_diag": np.array([1.0, 1.0], dtype=float),
    "u_nom_tgt": np.zeros(B_aug.shape[1], dtype=float),
    "w_x_tgt": 1e-6,
}

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92


In [ ]:
LMPC_first_step = FirstStepContractionTrackingLyapunovMpcSolver(
    A_aug=A_aug,
    B_aug=B_aug,
    C_aug=C_aug,
    Qy_diag=Qy_diag_std,
    Su_diag=Su_diag_std,
    NP=predict_h,
    NC=cont_h,
    P_x=P_x,
    K_x=K_x,
    Rdu_diag=Rdu_diag_std,
    terminal_set_on=run_config["terminal_set_on"],
    terminal_alpha_scale=run_config["terminal_alpha_scale"],
    terminal_cost_scale=run_config["terminal_cost_scale"],
)

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)

results_first_step = run_standard_tracking_lyapunov_mpc_first_step_contraction(
    system=cstr,
    LMPC_obj=LMPC_first_step,
    y_sp_scenario=y_sp_scenario,
    n_tests=n_tests,
    set_points_len=set_points_len,
    steady_states=steady_states,
    IC_opt=IC_opt,
    bnds=bnds,
    L=L,
    data_min=data_min,
    data_max=data_max,
    test_cycle=TEST_CYCLE,
    reward_fn=reward_fn,
    nominal_qi=nominal_qi,
    nominal_qs=nominal_qs,
    nominal_ha=nominal_hA,
    qi_change=qi_change,
    qs_change=qs_change,
    ha_change=ha_change,
    Qs_tgt_diag=run_config["Qs_tgt_diag"],
    Ru_tgt_diag=run_config["Ru_tgt_diag"],
    u_nom_tgt=run_config["u_nom_tgt"],
    w_x_tgt=run_config["w_x_tgt"],
    mode="nominal",
    use_external_target_for_tracking=run_config["use_external_target_for_tracking"],
    disturbance_after_step=run_config["disturbance_after_step"],
    skip_terminal_if_alpha_small=run_config["skip_terminal_if_alpha_small"],
    alpha_terminal_min=run_config["alpha_terminal_min"],
    use_target_on_solver_fail=run_config["use_target_on_solver_fail"],
    rho_lyap=run_config["rho_lyap"],
    eps_lyap=run_config["eps_lyap"],
    first_step_contraction_on=run_config["first_step_contraction_on"],
)

results_first_step[2][-1] if results_first_step[2] else None


In [ ]:
last_info = results_first_step[11][-1] if results_first_step[11] else {}
summary = {
    "method": last_info.get("method"),
    "success": last_info.get("success"),
    "alpha_terminal": last_info.get("alpha_terminal"),
    "V_k": last_info.get("V_k"),
    "V_next_first": last_info.get("V_next_first"),
    "V_bound": last_info.get("V_bound"),
    "contraction_margin": last_info.get("contraction_margin"),
    "first_step_contraction_satisfied": last_info.get("first_step_contraction_satisfied"),
}
summary
